# DecodeLabs | Data Analytics - Project 3  
## SQL Data Analysis

**Dataset:** Cleaned_Dataset.xlsx, output from Project 1  
**SQL Engine:** SQLite3 using Python `sqlite3`  
**Output Files:** SQL_Query_Results.xlsx and queries.sql  

## Project Objective

This project uses SQL queries to extract business insights from a cleaned e-commerce transactions dataset. The analysis demonstrates SQL fundamentals including SELECT, WHERE, ORDER BY, GROUP BY, and aggregation functions such as COUNT, SUM, and AVG.

## Required SQL Skills Demonstrated

- SELECT statements  
- WHERE filtering  
- ORDER BY sorting  
- GROUP BY categorical grouping  
- COUNT, SUM, and AVG aggregations  
- HAVING for grouped-data filtering  
- Combined multi-clause SQL queries

In [1]:
import sqlite3
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


In [2]:
# ─────────────────────────────────────────────
# STEP 0 — Setup and LOAD THE CLEANED DATASET INTO SQLITE
# ─────────────────────────────────────────────
print("=" * 55)
print("  STEP 0 — LOADING CLEANED DATASET INTO SQL ENGINE")
print("=" * 55)

df = pd.read_excel("Cleaned_Dataset.xlsx")
print(f"  + Dataset loaded successfully")
print(f"  + Shape  : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  + Columns: {', '.join(df.columns.tolist())}\n")

# Build an in-memory SQLite database and load the dataframe as a table
conn = sqlite3.connect(":memory:")
df.to_sql("orders", conn, index=False, if_exists="replace")
cursor = conn.cursor()

print("  + SQLite in-memory database created")
print("  + Table 'orders' populated with all 1,200 records\n")


# Helper to run a query, print a preview, and return the full result as a DataFrame
def run_query(title, sql, preview_rows=10):
    print("-" * 55)
    print(f"  {title}")
    print("-" * 55)
    print(sql.strip())
    print()
    result = pd.read_sql_query(sql, conn)
    print(result.head(preview_rows).to_string(index=False))
    print(f"\n  -> {len(result)} row(s) returned\n")
    return result


# Collect every query + result set for the Excel + Word deliverables
results = {}
queries_sql = {}

  STEP 0 — LOADING CLEANED DATASET INTO SQL ENGINE
  + Dataset loaded successfully
  + Shape  : 1200 rows x 14 columns
  + Columns: OrderID, Date, CustomerID, Product, Quantity, UnitPrice, ShippingAddress, PaymentMethod, OrderStatus, TrackingNumber, ItemsInCart, CouponCode, ReferralSource, TotalPrice

  + SQLite in-memory database created
  + Table 'orders' populated with all 1,200 records



## Step 1: Verify SQL Table Structure

The cleaned dataset was loaded into an SQLite table named `orders`. The schema check confirms the available fields before writing analytical SQL queries.

In [3]:
# ----------------------------------------------------------------------
# STEP 1 — VERIFY SQL TABLE STRUCTURE
# ----------------------------------------------------------------------

schema_sql = """
PRAGMA table_info(orders);
"""

schema_df = pd.read_sql_query(schema_sql, conn)
schema_df

,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TEXT,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


## Step 2: Dataset Validation

This validation query confirms the number of rows, unique orders, total revenue, and average order value in the SQL table. This step improves reliability before moving into business queries.

In [4]:
# ----------------------------------------------------------------------
# STEP 2 — BASIC TABLE VALIDATION
# ----------------------------------------------------------------------

validation_sql = """
SELECT 
    COUNT(*) AS TotalRows,
    COUNT(DISTINCT OrderID) AS UniqueOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders;
"""

validation_result = run_query(
    "Dataset validation: total rows, unique orders, total revenue and average order value",
    validation_sql
)

results["Validation_Summary"] = validation_result
queries_sql["Validation_Summary"] = validation_sql

-------------------------------------------------------
  Dataset validation: total rows, unique orders, total revenue and average order value
-------------------------------------------------------
SELECT 
    COUNT(*) AS TotalRows,
    COUNT(DISTINCT OrderID) AS UniqueOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders;

 TotalRows  UniqueOrders  TotalRevenue  AverageOrderValue
      1200          1200    1264761.96            1053.97

  -> 1 row(s) returned



## Query 1: Basic SELECT — First 10 Orders

**Business question:** What does the order-level dataset look like?

**SQL concept used:** SELECT

**Purpose:** This query displays a sample of order records to understand the key fields available for analysis.

In [5]:
# ─────────────────────────────────────────────
# QUERY 1 — BASIC SELECT
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 1 — BASIC SELECT")
print("=" * 55)

sql_1 = """
SELECT OrderID, CustomerID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
LIMIT 10;
"""
results["Q1_Basic_Select"] = run_query("Q1: First 10 orders (basic SELECT)", sql_1)
queries_sql["Q1_Basic_Select"] = sql_1



  QUERY 1 — BASIC SELECT
-------------------------------------------------------
  Q1: First 10 orders (basic SELECT)
-------------------------------------------------------
SELECT OrderID, CustomerID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
LIMIT 10;

  OrderID CustomerID Product  Quantity  UnitPrice  TotalPrice
ORD200000     C72649 Monitor         5     570.62     2853.10
ORD200001     C75739   Phone         2     151.35      302.70
ORD200002     C81728  Tablet         5     550.68     2753.40
ORD200003     C33540   Chair         1     273.19      273.19
ORD200004     C81840 Printer         4     626.01     2504.04
ORD200005     C37249   Phone         2     245.86      491.72
ORD200006     C83492  Laptop         1     664.42      664.42
ORD200007     C41460 Monitor         5     149.55      747.75
ORD200008     C26817   Phone         2     134.28      268.56
ORD200009     C31946    Desk         4     509.38     2037.52

  -> 10 row(s) returned



**Insight:** The dataset contains transaction-level information such as order ID, customer ID, product, quantity, unit price, and total price. These fields can support product, revenue, and customer-order analysis.

## Query 2: WHERE Equality — Cancelled Orders

**Business question:** Which orders were cancelled?

**SQL concept used:** WHERE with equality condition

**Purpose:** This query filters the dataset to show only orders where `OrderStatus = 'Cancelled'`.

In [6]:
# ─────────────────────────────────────────────
# QUERY 2 — WHERE: EQUALITY FILTER
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 2 — WHERE (EQUALITY)")
print("=" * 55)

sql_2 = """
SELECT OrderID, CustomerID, Product, OrderStatus, TotalPrice
FROM orders
WHERE OrderStatus = 'Cancelled';
"""
results["Q2_Where_Equality"] = run_query(
    "Q2: All cancelled orders (WHERE equality)", sql_2
)
queries_sql["Q2_Where_Equality"] = sql_2



  QUERY 2 — WHERE (EQUALITY)
-------------------------------------------------------
  Q2: All cancelled orders (WHERE equality)
-------------------------------------------------------
SELECT OrderID, CustomerID, Product, OrderStatus, TotalPrice
FROM orders
WHERE OrderStatus = 'Cancelled';

  OrderID CustomerID Product OrderStatus  TotalPrice
ORD200002     C81728  Tablet   Cancelled     2753.40
ORD200008     C26817   Phone   Cancelled      268.56
ORD200023     C42372   Phone   Cancelled     1093.00
ORD200033     C52184   Chair   Cancelled      162.77
ORD200042     C28712 Printer   Cancelled      303.32
ORD200043     C60572   Chair   Cancelled     1497.63
ORD200050     C96494  Tablet   Cancelled      179.22
ORD200055     C95984 Printer   Cancelled     1033.20
ORD200056     C54298  Tablet   Cancelled     1437.63
ORD200058     C60190   Phone   Cancelled     1898.76

  -> 250 row(s) returned



**Insight:** Cancelled orders can reduce realised revenue and may indicate fulfilment, customer satisfaction, or payment-related issues. Management should monitor cancellation patterns by product and customer segment.

## Query 3: WHERE Comparison — High-Value Orders

**Business question:** Which orders have a value of at least $2,000?

**SQL concept used:** WHERE with comparison condition and ORDER BY

**Purpose:** This query identifies high-value orders and sorts them from highest to lowest order value.

In [7]:
# ─────────────────────────────────────────────
# QUERY 3 — WHERE: COMPARISON FILTER
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 3 — WHERE (COMPARISON)")
print("=" * 55)

sql_3 = """
SELECT OrderID, CustomerID, Product, Quantity, TotalPrice
FROM orders
WHERE TotalPrice >= 2000
ORDER BY TotalPrice DESC;
"""
results["Q3_Where_Comparison"] = run_query(
    "Q3: High-value orders >= $2,000 (WHERE comparison)", sql_3
)
queries_sql["Q3_Where_Comparison"] = sql_3



  QUERY 3 — WHERE (COMPARISON)
-------------------------------------------------------
  Q3: High-value orders >= $2,000 (WHERE comparison)
-------------------------------------------------------
SELECT OrderID, CustomerID, Product, Quantity, TotalPrice
FROM orders
WHERE TotalPrice >= 2000
ORDER BY TotalPrice DESC;

  OrderID CustomerID Product  Quantity  TotalPrice
ORD200789     C57276  Tablet         5     3456.40
ORD201122     C38840 Monitor         5     3390.95
ORD200632     C67260  Laptop         5     3390.80
ORD200469     C13877   Chair         5     3384.90
ORD200328     C18404  Tablet         5     3370.20
ORD200107     C16775 Printer         5     3353.75
ORD200326     C65986  Laptop         5     3352.40
ORD201065     C47778 Printer         5     3334.00
ORD201031     C59183   Phone         5     3322.55
ORD200463     C25276  Laptop         5     3313.90

  -> 180 row(s) returned



**Insight:** High-value orders show which products and customers contribute most to revenue. These orders may be useful for premium customer targeting or high-value product promotion.

## Query 4: WHERE LIKE — Laptop Orders

**Business question:** Which transactions involve laptop products?

**SQL concept used:** WHERE with LIKE pattern matching

**Purpose:** This query demonstrates pattern matching by filtering product names that contain the word “Laptop”.

In [8]:
# ─────────────────────────────────────────────
# QUERY 4 — WHERE: PATTERN MATCHING (LIKE)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 4 — WHERE (PATTERN MATCHING)")
print("=" * 55)

sql_4 = """
SELECT OrderID, CustomerID, Product, TotalPrice
FROM orders
WHERE Product LIKE '%Laptop%'
ORDER BY TotalPrice DESC;
"""
results["Q4_Where_LikePattern"] = run_query(
    "Q4: Laptop orders using WHERE LIKE pattern matching",
    sql_4
)
queries_sql["Q4_Where_LikePattern"] = sql_4


  QUERY 4 — WHERE (PATTERN MATCHING)
-------------------------------------------------------
  Q4: Laptop orders using WHERE LIKE pattern matching
-------------------------------------------------------
SELECT OrderID, CustomerID, Product, TotalPrice
FROM orders
WHERE Product LIKE '%Laptop%'
ORDER BY TotalPrice DESC;

  OrderID CustomerID Product  TotalPrice
ORD200632     C67260  Laptop     3390.80
ORD200326     C65986  Laptop     3352.40
ORD200463     C25276  Laptop     3313.90
ORD200367     C13108  Laptop     3293.85
ORD200540     C87281  Laptop     3243.25
ORD200764     C35983  Laptop     3137.15
ORD200492     C39074  Laptop     3032.60
ORD200633     C79533  Laptop     3008.60
ORD201087     C84134  Laptop     2772.28
ORD201156     C20512  Laptop     2763.12

  -> 173 row(s) returned



**Insight:** Pattern matching is useful when product names or text fields need flexible filtering. Laptop orders can be reviewed separately to understand performance in this product category.

## Query 5: ORDER BY — Top 10 Highest-Value Orders

**Business question:** What are the highest-value orders in the dataset?

**SQL concept used:** ORDER BY with LIMIT

**Purpose:** This query sorts orders by total value in descending order and returns the top 10 transactions.

In [9]:
# ─────────────────────────────────────────────
# QUERY 5 — ORDER BY
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 5 — ORDER BY")
print("=" * 55)

sql_5 = """
SELECT OrderID, CustomerID, Product, TotalPrice, Date
FROM orders
ORDER BY TotalPrice DESC
LIMIT 10;
"""
results["Q5_OrderBy_TopOrders"] = run_query(
    "Q5: Top 10 highest-value orders (ORDER BY)", sql_5
)
queries_sql["Q5_OrderBy_TopOrders"] = sql_5



  QUERY 5 — ORDER BY
-------------------------------------------------------
  Q5: Top 10 highest-value orders (ORDER BY)
-------------------------------------------------------
SELECT OrderID, CustomerID, Product, TotalPrice, Date
FROM orders
ORDER BY TotalPrice DESC
LIMIT 10;

  OrderID CustomerID Product  TotalPrice       Date
ORD200789     C57276  Tablet     3456.40 2023-08-17
ORD201122     C38840 Monitor     3390.95 2023-06-07
ORD200632     C67260  Laptop     3390.80 2023-05-02
ORD200469     C13877   Chair     3384.90 2023-11-26
ORD200328     C18404  Tablet     3370.20 2023-02-28
ORD200107     C16775 Printer     3353.75 2023-03-27
ORD200326     C65986  Laptop     3352.40 2024-07-01
ORD201065     C47778 Printer     3334.00 2023-10-30
ORD201031     C59183   Phone     3322.55 2023-02-28
ORD200463     C25276  Laptop     3313.90 2023-05-26

  -> 10 row(s) returned



**Insight:** The highest-value orders identify the most financially important transactions. These can help the business understand which products or customer orders drive exceptional revenue.

## Query 6: GROUP BY + COUNT — Orders per Product

**Business question:** Which products receive the highest number of orders?

**SQL concept used:** GROUP BY and COUNT

**Purpose:** This query groups orders by product and counts the number of orders for each product.

In [10]:
# ─────────────────────────────────────────────
# QUERY 6 — GROUP BY + COUNT
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 6 — GROUP BY + COUNT")
print("=" * 55)

sql_6 = """
SELECT Product, COUNT(*) AS OrderCount
FROM orders
GROUP BY Product
ORDER BY OrderCount DESC;
"""
results["Q6_GroupBy_Count"] = run_query(
    "Q6: Number of orders per product (GROUP BY + COUNT)", sql_6
)
queries_sql["Q6_GroupBy_Count"] = sql_6


  QUERY 6 — GROUP BY + COUNT
-------------------------------------------------------
  Q6: Number of orders per product (GROUP BY + COUNT)
-------------------------------------------------------
SELECT Product, COUNT(*) AS OrderCount
FROM orders
GROUP BY Product
ORDER BY OrderCount DESC;

Product  OrderCount
Printer         181
 Tablet         179
  Chair         178
 Laptop         173
   Desk         170
Monitor         163
  Phone         156

  -> 7 row(s) returned



**Insight:** Product order counts show demand volume. High-order products may require stronger stock planning, while low-order products may need pricing or promotion review.

## Query 7: GROUP BY + SUM — Revenue by Payment Method

**Business question:** Which payment method generates the highest total revenue?

**SQL concept used:** GROUP BY and SUM

**Purpose:** This query groups transactions by payment method and calculates total revenue for each method.

In [11]:
# ─────────────────────────────────────────────
# QUERY 7 — GROUP BY + SUM
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 7 — GROUP BY + SUM")
print("=" * 55)

sql_7 = """
SELECT PaymentMethod, SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC;
"""
results["Q7_GroupBy_Sum"] = run_query(
    "Q7: Total revenue per payment method (GROUP BY + SUM)", sql_7
)
queries_sql["Q7_GroupBy_Sum"] = sql_7


  QUERY 7 — GROUP BY + SUM
-------------------------------------------------------
  Q7: Total revenue per payment method (GROUP BY + SUM)
-------------------------------------------------------
SELECT PaymentMethod, SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC;

PaymentMethod  TotalRevenue
  Credit Card     263847.63
       Online     262442.94
         Cash     259786.29
    Gift Card     246323.92
   Debit Card     232361.18

  -> 5 row(s) returned



**Insight:** Revenue by payment method helps identify how customers prefer to pay and whether particular payment channels are associated with stronger sales value.

## Query 8: GROUP BY + AVG — Average Order Value by Referral Source

**Business question:** Which referral source brings the highest-value orders on average?

**SQL concept used:** GROUP BY and AVG

**Purpose:** This query calculates the average order value for each referral source.

In [12]:
# ─────────────────────────────────────────────
# QUERY 8 — GROUP BY + AVG
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 8 — GROUP BY + AVG")
print("=" * 55)

sql_8 = """
SELECT ReferralSource, ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
GROUP BY ReferralSource
ORDER BY AvgOrderValue DESC;
"""
results["Q8_GroupBy_Avg"] = run_query(
    "Q8: Average order value per referral source (GROUP BY + AVG)", sql_8
)
queries_sql["Q8_GroupBy_Avg"] = sql_8



  QUERY 8 — GROUP BY + AVG
-------------------------------------------------------
  Q8: Average order value per referral source (GROUP BY + AVG)
-------------------------------------------------------
SELECT ReferralSource, ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
GROUP BY ReferralSource
ORDER BY AvgOrderValue DESC;

ReferralSource  AvgOrderValue
      Facebook        1098.29
     Instagram        1062.88
         Email        1047.23
        Google        1039.18
      Referral        1021.69

  -> 5 row(s) returned



**Insight:** Facebook has the highest average order value, which suggests that it may attract higher-value customers. Referral sources should therefore be compared by value, not only by order count.

## Query 9: HAVING — Products Above $150,000 Revenue

**Business question:** Which products generate more than $150,000 in total revenue?

**SQL concept used:** GROUP BY, SUM, COUNT, and HAVING

**Purpose:** This query filters grouped product revenue using HAVING, which is used after GROUP BY.

In [13]:
# ─────────────────────────────────────────────
# QUERY 9 — HAVING (filtering grouped data)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 9 — HAVING")
print("=" * 55)

sql_9 = """
SELECT Product, SUM(TotalPrice) AS TotalRevenue, COUNT(*) AS OrderCount
FROM orders
GROUP BY Product
HAVING SUM(TotalPrice) > 150000
ORDER BY TotalRevenue DESC;
"""
results["Q9_Having_RevenueFilter"] = run_query(
    "Q9: Products generating over $150,000 in revenue (HAVING)", sql_9
)
queries_sql["Q9_Having_RevenueFilter"] = sql_9



  QUERY 9 — HAVING
-------------------------------------------------------
  Q9: Products generating over $150,000 in revenue (HAVING)
-------------------------------------------------------
SELECT Product, SUM(TotalPrice) AS TotalRevenue, COUNT(*) AS OrderCount
FROM orders
GROUP BY Product
HAVING SUM(TotalPrice) > 150000
ORDER BY TotalRevenue DESC;

Product  TotalRevenue  OrderCount
  Chair     195620.11         178
Printer     195612.61         181
 Laptop     192126.56         173
 Tablet     186568.95         179
Monitor     175651.41         163
   Desk     167459.93         170
  Phone     151722.39         156

  -> 7 row(s) returned



**Insight:** The HAVING query identifies the strongest revenue-generating products. These products should be prioritised for stock availability, campaign planning, and product-level monitoring.

## Query 10: Combined SQL Query — Coupon Performance for Delivered Orders

**Business question:** Among delivered orders, which coupon code has the highest average order value?

**SQL concept used:** WHERE, GROUP BY, COUNT, AVG, and ORDER BY

**Purpose:** This query filters delivered orders, groups them by coupon code, counts delivered orders, and calculates average order value.

In [14]:
# ─────────────────────────────────────────────
# QUERY 10 — COMBINED: WHERE + GROUP BY + ORDER BY
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  QUERY 10 — COMBINED MULTI-CLAUSE QUERY")
print("=" * 55)

sql_10 = """
SELECT CouponCode, COUNT(*) AS OrdersDelivered, ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY CouponCode
ORDER BY AvgOrderValue DESC;
"""
results["Q10_Combined_Multi"] = run_query(
    "Q10: Delivered-order performance by coupon code (WHERE + GROUP BY + ORDER BY)",
    sql_10,
)
queries_sql["Q10_Combined_Multi"] = sql_10



  QUERY 10 — COMBINED MULTI-CLAUSE QUERY
-------------------------------------------------------
  Q10: Delivered-order performance by coupon code (WHERE + GROUP BY + ORDER BY)
-------------------------------------------------------
SELECT CouponCode, COUNT(*) AS OrdersDelivered, ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY CouponCode
ORDER BY AvgOrderValue DESC;

CouponCode  OrdersDelivered  AvgOrderValue
    SAVE10               63        1104.04
  FREESHIP               61        1074.33
      NONE               52        1028.35
  WINTER15               55         982.50

  -> 4 row(s) returned



**Insight:** SAVE10 produced the highest average order value among delivered coupon-code groups, followed by FREESHIP. This suggests that coupon performance should be judged by both order volume and order value.

## Step 3: Export SQL Query Results to Excel

All SQL query outputs are exported into a single Excel workbook. Each query result is saved in a separate sheet, making the analysis easy to review and share.

In [15]:
# ----------------------------------------------------------------------
# STEP 3 — EXPORT SQL QUERY RESULTS TO EXCEL
# ----------------------------------------------------------------------

output_excel = "SQL_Query_Results.xlsx"

# Check whether results dictionary has query outputs
if len(results) == 0:
    print("No query results found in the results dictionary.")
    print("Please make sure each query is saved into results before exporting.")
else:
    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        for sheet_name, result_df in results.items():
            safe_sheet_name = sheet_name[:31]
            result_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

    print(f"All SQL query results have been exported to: {output_excel}")

All SQL query results have been exported to: SQL_Query_Results.xlsx


## Step 4: Write Standalone SQL Script

All SQL queries are saved into a separate `.sql` file. This file can be reused in any SQL environment and shows the full query logic used in the project.

In [17]:
# ----------------------------------------------------------------------
# STEP 4 — WRITE STANDALONE SQL SCRIPT
# ----------------------------------------------------------------------

sql_script_file = "queries.sql"

with open(sql_script_file, "w") as file:
    file.write("-- DecodeLabs Data Analytics Project 3\n")
    file.write("-- SQL Data Analysis Queries\n\n")

    for query_name, query_text in queries_sql.items():
        file.write(f"-- {query_name}\n")
        file.write(query_text.strip())
        file.write("\n\n")

print(f"All SQL queries have been saved to: {sql_script_file}")

All SQL queries have been saved to: queries.sql


## Step 5: Final Executive Summary

This SQL analysis successfully demonstrates the main requirements of Project 3. The notebook uses SELECT queries to inspect transaction records, WHERE clauses to filter specific business cases, ORDER BY to rank results, GROUP BY to summarise categories, and aggregation functions such as COUNT, SUM, and AVG to convert raw order-level data into business insights.

The analysis shows product demand through order counts, revenue performance by payment method, average order value by referral source, and coupon performance for delivered orders. The extended HAVING query adds further value by filtering grouped results to identify high-revenue products.

Overall, the project demonstrates the use of SQL fundamentals to transform a cleaned e-commerce dataset into structured business intelligence. The exported Excel workbook and standalone SQL script also make the outputs easy to review, reuse, and share.